In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2

In [2]:
# -----------------------------------
# 1. Load Cleaned Dataset
# -----------------------------------
df = pd.read_excel("cleaned_titanic_dataset.xlsx")

print("Original Dataset:")
print(df.head())

Original Dataset:
   sn  pclass  survived                         name  gender   age  family  \
0   1       3         0                  Mr. Anthony    male  42.0       0   
1   2       3         0        Master. Eugene Joseph    male  26.5       2   
2   3       2         0  Abbott, Mr. Rossmore Edward    male  26.5       2   
3   4       3         1  Abbott, Mr. Rossmore Edward  female  35.0       2   
4   5       3         1  Abelseth, Miss. Karen Marie  female  16.0       0   

    fare embarked       date  
0   7.55        S 1990-01-01  
1  20.25        S 1990-01-02  
2    NaN        S 1990-01-03  
3  20.25        S 1990-01-04  
4   7.65        S 1990-01-05  


In [3]:
# -----------------------------------
# 2. Feature Engineering
# -----------------------------------

# Create new feature: Family Size
df["family_size"] = df["family"] + 1

# Create Age Group feature
def age_group(age):
    if age < 18:
        return "Child"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

df["age_group"] = df["age"].apply(age_group)

print("\nDataset after Feature Engineering:")
print(df.head())


Dataset after Feature Engineering:
   sn  pclass  survived                         name  gender   age  family  \
0   1       3         0                  Mr. Anthony    male  42.0       0   
1   2       3         0        Master. Eugene Joseph    male  26.5       2   
2   3       2         0  Abbott, Mr. Rossmore Edward    male  26.5       2   
3   4       3         1  Abbott, Mr. Rossmore Edward  female  35.0       2   
4   5       3         1  Abelseth, Miss. Karen Marie  female  16.0       0   

    fare embarked       date  family_size age_group  
0   7.55        S 1990-01-01            1     Adult  
1  20.25        S 1990-01-02            3     Adult  
2    NaN        S 1990-01-03            3     Adult  
3  20.25        S 1990-01-04            3     Adult  
4   7.65        S 1990-01-05            1     Child  


In [4]:
# -----------------------------------
# 3. Handle Categorical Variables
# -----------------------------------

# Label Encoding
encoder = LabelEncoder()

df["gender"] = encoder.fit_transform(df["gender"])
df["embarked"] = encoder.fit_transform(df["embarked"])
df["age_group"] = encoder.fit_transform(df["age_group"])

print("\nAfter Encoding:")
print(df.head())


After Encoding:
   sn  pclass  survived                         name  gender   age  family  \
0   1       3         0                  Mr. Anthony       1  42.0       0   
1   2       3         0        Master. Eugene Joseph       1  26.5       2   
2   3       2         0  Abbott, Mr. Rossmore Edward       1  26.5       2   
3   4       3         1  Abbott, Mr. Rossmore Edward       0  35.0       2   
4   5       3         1  Abelseth, Miss. Karen Marie       0  16.0       0   

    fare  embarked       date  family_size  age_group  
0   7.55         1 1990-01-01            1          0  
1  20.25         1 1990-01-02            3          0  
2    NaN         1 1990-01-03            3          0  
3  20.25         1 1990-01-04            3          0  
4   7.65         1 1990-01-05            1          1  


In [5]:
# -----------------------------------
# 4. Select Important Features
# -----------------------------------

# Features (X)
X = df[[
    "pclass",
    "gender",
    "age",
    "family",
    "fare",
    "embarked",
    "family_size",
    "age_group"
]]

# Target (y)
y = df["survived"]

In [6]:
# -----------------------------------
# 5. Feature Scaling
# -----------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("\nScaled Features:")
print(X_scaled[:5])


Scaled Features:
[[ 0.65465367  0.81649658  2.05633274 -0.91766294 -0.88801673  0.65465367
  -0.91766294 -0.33333333]
 [ 0.65465367  0.81649658 -0.02688017  1.3764944   0.84542532  0.65465367
   1.3764944  -0.33333333]
 [-1.52752523  0.81649658 -0.02688017  1.3764944          nan  0.65465367
   1.3764944  -0.33333333]
 [ 0.65465367 -1.22474487  1.11552691  1.3764944   0.84542532  0.65465367
   1.3764944  -0.33333333]
 [ 0.65465367 -1.22474487 -1.43808891 -0.91766294 -0.87436758  0.65465367
  -0.91766294  3.        ]]


In [10]:
# -----------------------------------
# 6. Handle Remaining Missing Values
# -----------------------------------

# Check missing values
print(X.isnull().sum())

# Fill missing values
X["age"] = X["age"].fillna(X["age"].median())
X["fare"] = X["fare"].fillna(X["fare"].median())

# For categorical columns
X["embarked"] = X["embarked"].fillna(X["embarked"].mode()[0])

# Final check
print("\nMissing Values After Filling:")
print(X.isnull().sum())

# -----------------------------------
# 7. Feature Selection
# -----------------------------------

selector = SelectKBest(score_func=chi2, k=5)

selected_features = selector.fit(X, y)

scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selected_features.scores_
})

print("\nFeature Scores:")
print(scores.sort_values(by="Score", ascending=False))

# -----------------------------------
# 8. Best Features
# -----------------------------------

best_features = X.columns[selected_features.get_support()]

print("\nSelected Best Features:")
print(best_features)

pclass         0
gender         0
age            0
family         0
fare           1
embarked       0
family_size    0
age_group      0
dtype: int64

Missing Values After Filling:
pclass         0
gender         0
age            0
family         0
fare           0
embarked       0
family_size    0
age_group      0
dtype: int64

Feature Scores:
       Feature     Score
2          age  5.169164
1       gender  1.777778
3       family  1.687500
4         fare  1.098534
6  family_size  0.750000
7    age_group  0.666667
0       pclass  0.098765
5     embarked  0.023810

Selected Best Features:
Index(['gender', 'age', 'family', 'fare', 'family_size'], dtype='object')


/tmp/ipykernel_2063/3566115576.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["age"] = X["age"].fillna(X["age"].median())
/tmp/ipykernel_2063/3566115576.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["fare"] = X["fare"].fillna(X["fare"].median())
/tmp/ipykernel_2063/3566115576.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org